# Zerlaut simulation plots

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import json5 as json

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from aux.environment import resolve_directory
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results

**Set main directory paths**

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

## Load and plot a simulation

In [ ]:
OUTPUT_DIR = WORK_DIR / "results" / "exp1"
TRIAL_IDS = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15] #201
CONFIG_PATH = OUTPUT_DIR / "config.json"
SKIP_VALUES = 8

# PLOT_STATES = None
PLOT_STATES = ["E", "I"]

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    loaded_config = json.load(config_file)

In [ ]:
from IPython.display import clear_output, display

n_rows = (len(TRIAL_IDS) + 3) // 4
fig, axes = plt.subplots(
    n_rows, 4, figsize=(24, 6 * max(n_rows, 4)), squeeze=False, constrained_layout=True,
)

plt.close(fig)
for ax in axes.flat[len(TRIAL_IDS):]:
    ax.set_axis_off()

for simulation_id, ax in zip(TRIAL_IDS, axes.flat):
    OUTPUT_PATH = OUTPUT_DIR / f"trial_{simulation_id:08d}.npz"

    with np.load(OUTPUT_PATH, allow_pickle=False) as saved:
        time_ms = np.asarray(saved["time_ms"])
        states = {name: np.asarray(saved[name]) for name in saved.files
                  if name not in ("time_ms", "metadata_json")}
        trial_metadata = json.loads(saved["metadata_json"].item())

    metadata = ResultMetadata(
        simulation=SimulationMetadata(
            **loaded_config["simulation"], output_dtype=loaded_config["output"]["dtype"],
        ),
        model=ModelMetadata(
            name=loaded_config["model"]["name"],
            parameters=trial_metadata["parameters"],
            output_variables=list(states),
            state_units={
                "E": "kHz", "I": "kHz",
                "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
                "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
            },
        ),
        stimulus=trial_metadata["stimulus"],
        stimulus_seed=trial_metadata["stimulus_seed"],
    )

    time_ms = time_ms[::SKIP_VALUES]
    states = {key: value[::SKIP_VALUES] for key, value in states.items()}
    fig, trial_axes = plot_results(
        time_ms, states, metadata,
        show_stimulus=True, ax=ax,
        plot_states=PLOT_STATES,
    )
    trial_axes[0].set_title(f"Trial {simulation_id}")
    clear_output(wait=True)
    display(fig)